In [19]:
%pip install pandas numpy matplotlib seaborn scipy sqlalchemy pymysql streamlit plotly python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import pandas as pd

raw_path = "../data/raw/"

customers   = pd.read_csv(raw_path + "olist_customers_dataset.csv")
geolocation = pd.read_csv(raw_path + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(raw_path + "olist_order_items_dataset.csv")
payments    = pd.read_csv(raw_path + "olist_order_payments_dataset.csv")
reviews     = pd.read_csv(raw_path + "olist_order_reviews_dataset.csv")
orders      = pd.read_csv(raw_path + "olist_orders_dataset.csv")
products    = pd.read_csv(raw_path + "olist_products_dataset.csv")
sellers     = pd.read_csv(raw_path + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(raw_path + "product_category_name_translation.csv")

print("All 9 files loaded!")

All 9 files loaded!


In [21]:
tables = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation,
}

for name, df in tables.items():
    print(f"{name:22} rows: {df.shape[0]:>8}   columns: {df.shape[1]}")

customers              rows:    99441   columns: 5
geolocation            rows:  1000163   columns: 5
order_items            rows:   112650   columns: 7
payments               rows:   103886   columns: 5
reviews                rows:    99224   columns: 7
orders                 rows:    99441   columns: 8
products               rows:    32951   columns: 9
sellers                rows:     3095   columns: 4
category_translation   rows:       71   columns: 2


In [22]:
for name, df in tables.items():
    print(f"--- {name} ---")
    missing = df.isnull().sum()
    print("Missing values:")
    print(missing[missing > 0] if missing.sum() > 0 else "None")
    print("Duplicate rows:", df.duplicated().sum())
    print()

--- customers ---
Missing values:
None
Duplicate rows: 0

--- geolocation ---
Missing values:
None
Duplicate rows: 261831

--- order_items ---
Missing values:
None
Duplicate rows: 0

--- payments ---
Missing values:
None
Duplicate rows: 0

--- reviews ---
Missing values:
review_comment_title      87656
review_comment_message    58247
dtype: int64
Duplicate rows: 0

--- orders ---
Missing values:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
Duplicate rows: 0

--- products ---
Missing values:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
Duplicate rows: 0

--- sellers ---
Missing values:
None
Duplicate rows: 0

--- category_translation ---
Missing values:
None
Duplicate ro

In [23]:
for name in ["reviews", "orders"]:
    df = tables[name]
    print(f"--- {name} ---")
    missing = df.isnull().sum()
    print(missing[missing > 0])
    print("Duplicate rows:", df.duplicated().sum())
    print()

--- reviews ---
review_comment_title      87656
review_comment_message    58247
dtype: int64
Duplicate rows: 0

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
Duplicate rows: 0



In [24]:
for name in ["products", "sellers", "category_translation"]:
    df = tables[name]
    print(f"--- {name} ---")
    missing = df.isnull().sum()
    print(missing[missing > 0])
    print("Duplicate rows:", df.duplicated().sum())
    print()

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
Duplicate rows: 0

--- sellers ---
Series([], dtype: int64)
Duplicate rows: 0

--- category_translation ---
Series([], dtype: int64)
Duplicate rows: 0



In [25]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [26]:
# Convert text to real dates. errors="coerce" turns bad/missing values into NaT
order_date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in order_date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [27]:
# Fix the "lenght" spelling mistake
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

# Missing category -> "unknown"
products["product_category_name"] = products["product_category_name"].fillna("unknown")

# Missing numbers -> median
num_cols = ["product_name_length", "product_description_length", "product_photos_qty",
            "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for col in num_cols:
    products[col] = products[col].fillna(products[col].median())

products.isnull().sum()

product_id                    0
product_category_name         0
product_name_length           0
product_description_length    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

In [28]:
reviews["review_comment_title"] = reviews["review_comment_title"].fillna("No comment")
reviews["review_comment_message"] = reviews["review_comment_message"].fillna("No comment")

print("Repeated review_id rows:", reviews["review_id"].duplicated().sum())

# The ER diagram says review_id is the primary key, so keep one row per review_id
reviews = reviews.drop_duplicates(subset="review_id", keep="first")
print("review_id unique now:", reviews["review_id"].is_unique)

Repeated review_id rows: 814
review_id unique now: True


In [29]:
geolocation = geolocation.drop_duplicates()

geolocation = (
    geolocation
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg({
        "geolocation_lat": "mean",
        "geolocation_lng": "mean",
        "geolocation_city": "first",
        "geolocation_state": "first",
    })
)
print(geolocation.shape)
print("zip prefix unique:", geolocation["geolocation_zip_code_prefix"].is_unique)

(19015, 5)
zip prefix unique: True


In [30]:
# Consistent text: lowercase/trimmed cities, uppercase states
for df, city_col, state_col in [
    (customers, "customer_city", "customer_state"),
    (sellers, "seller_city", "seller_state"),
    (geolocation, "geolocation_city", "geolocation_state"),
]:
    df[city_col] = df[city_col].str.strip().str.lower()
    df[state_col] = df[state_col].str.strip().str.upper()

# Invalid-value checks (these only print, nothing is deleted yet)
print("Items with price <= 0:", (order_items["price"] <= 0).sum())
print("Payments with value <= 0:", (payments["payment_value"] <= 0).sum())
print("Review scores outside 1-5:", (~reviews["review_score"].between(1, 5)).sum())
print("Delivered before purchase:",
      (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum())
print(payments["payment_type"].value_counts())

Items with price <= 0: 0
Payments with value <= 0: 9
Review scores outside 1-5: 0
Delivered before purchase: 0
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [31]:
import os
os.makedirs("../data/cleaned", exist_ok=True)

cleaned = {
    "customers": customers, "geolocation": geolocation, "order_items": order_items,
    "payments": payments, "reviews": reviews, "orders": orders,
    "products": products, "sellers": sellers, "category_translation": category_translation,
}
for name, df in cleaned.items():
    df.to_csv(f"../data/cleaned/{name}_clean.csv", index=False)
print("Cleaned files saved!")

Cleaned files saved!


In [32]:
payments[payments["payment_value"] <= 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [33]:
bad = payments[(payments["payment_value"] <= 0) | (payments["payment_type"] == "not_defined")]
orders[orders["order_id"].isin(bad["order_id"])]["order_status"].value_counts()

order_status
delivered    4
canceled     3
shipped      1
Name: count, dtype: int64

In [34]:
payments = payments[payments["payment_type"] != "not_defined"]
print(payments["payment_type"].value_counts())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
Name: count, dtype: int64


In [35]:
print("orders without customer:", (~orders["customer_id"].isin(customers["customer_id"])).sum())
print("items without order:", (~order_items["order_id"].isin(orders["order_id"])).sum())
print("items without product:", (~order_items["product_id"].isin(products["product_id"])).sum())
print("items without seller:", (~order_items["seller_id"].isin(sellers["seller_id"])).sum())
print("payments without order:", (~payments["order_id"].isin(orders["order_id"])).sum())
print("reviews without order:", (~reviews["order_id"].isin(orders["order_id"])).sum())

orders without customer: 0
items without order: 0
items without product: 0
items without seller: 0
payments without order: 0
reviews without order: 0


In [36]:
print("customers zip not in geolocation:",
      (~customers["customer_zip_code_prefix"].isin(geolocation["geolocation_zip_code_prefix"])).sum())
print("sellers zip not in geolocation:",
      (~sellers["seller_zip_code_prefix"].isin(geolocation["geolocation_zip_code_prefix"])).sum())

customers zip not in geolocation: 278
sellers zip not in geolocation: 7


In [37]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv("../.env")

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT DATABASE();")).fetchone())

OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on '22@localhost' ([Errno 11003] getaddrinfo failed)")
(Background on this error at: https://sqlalche.me/e/21/e3q8)

In [38]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv("../.env")

url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(url)

with engine.connect() as conn:
    print(conn.execute(text("SELECT DATABASE();")).fetchone())

('cart2insights_db',)


In [39]:
ddl_statements = [
"SET FOREIGN_KEY_CHECKS = 0",
"DROP TABLE IF EXISTS order_reviews, order_payments, order_items, orders, products, sellers, customers, geolocation, product_category_translation",
"SET FOREIGN_KEY_CHECKS = 1",

"""CREATE TABLE geolocation (
    geolocation_zip_code_prefix INT PRIMARY KEY,
    geolocation_lat DOUBLE,
    geolocation_lng DOUBLE,
    geolocation_city VARCHAR(100),
    geolocation_state CHAR(2)
)""",

"""CREATE TABLE customers (
    customer_id VARCHAR(50) PRIMARY KEY,
    customer_unique_id VARCHAR(50),
    customer_zip_code_prefix INT,
    customer_city VARCHAR(100),
    customer_state CHAR(2),
    INDEX idx_customer_unique (customer_unique_id)
)""",

"""CREATE TABLE sellers (
    seller_id VARCHAR(50) PRIMARY KEY,
    seller_zip_code_prefix INT,
    seller_city VARCHAR(100),
    seller_state CHAR(2)
)""",

"""CREATE TABLE product_category_translation (
    product_category_name VARCHAR(100) PRIMARY KEY,
    product_category_name_english VARCHAR(100)
)""",

"""CREATE TABLE products (
    product_id VARCHAR(50) PRIMARY KEY,
    product_category_name VARCHAR(100),
    product_name_length DOUBLE,
    product_description_length DOUBLE,
    product_photos_qty DOUBLE,
    product_weight_g DOUBLE,
    product_length_cm DOUBLE,
    product_height_cm DOUBLE,
    product_width_cm DOUBLE
)""",

"""CREATE TABLE orders (
    order_id VARCHAR(50) PRIMARY KEY,
    customer_id VARCHAR(50),
    order_status VARCHAR(20),
    order_purchase_timestamp DATETIME,
    order_approved_at DATETIME,
    order_delivered_carrier_date DATETIME,
    order_delivered_customer_date DATETIME,
    order_estimated_delivery_date DATETIME,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
)""",

"""CREATE TABLE order_items (
    order_id VARCHAR(50),
    order_item_id INT,
    product_id VARCHAR(50),
    seller_id VARCHAR(50),
    shipping_limit_date DATETIME,
    price DOUBLE,
    freight_value DOUBLE,
    PRIMARY KEY (order_id, order_item_id),
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id),
    FOREIGN KEY (seller_id) REFERENCES sellers(seller_id)
)""",

"""CREATE TABLE order_payments (
    order_id VARCHAR(50),
    payment_sequential INT,
    payment_type VARCHAR(20),
    payment_installments INT,
    payment_value DOUBLE,
    PRIMARY KEY (order_id, payment_sequential),
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
)""",

"""CREATE TABLE order_reviews (
    review_id VARCHAR(50) PRIMARY KEY,
    order_id VARCHAR(50),
    review_score INT,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date DATETIME,
    review_answer_timestamp DATETIME,
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
)""",
]

with engine.begin() as conn:
    for stmt in ddl_statements:
        conn.execute(text(stmt))

print("All 9 tables created!")

All 9 tables created!


In [40]:
load_plan = [
    ("geolocation", geolocation),
    ("customers", customers),
    ("sellers", sellers),
    ("product_category_translation", category_translation),
    ("products", products),
    ("orders", orders),
    ("order_items", order_items),
    ("order_payments", payments),
    ("order_reviews", reviews),
]

for table_name, df in load_plan:
    df = df.astype(object).where(df.notnull(), None)   # NaN/NaT -> NULL
    df.to_sql(table_name, engine, if_exists="append", index=False,
              chunksize=5000, method="multi")
    print(f"Loaded {table_name}: {len(df)} rows")

Loaded geolocation: 19015 rows
Loaded customers: 99441 rows
Loaded sellers: 3095 rows
Loaded product_category_translation: 71 rows
Loaded products: 32951 rows
Loaded orders: 99441 rows
Loaded order_items: 112650 rows
Loaded order_payments: 103883 rows
Loaded order_reviews: 98410 rows


In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv("../.env")
url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)

orders      = pd.read_sql("SELECT * FROM orders", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
customers   = pd.read_sql("SELECT * FROM customers", engine)
reviews     = pd.read_sql("SELECT * FROM order_reviews", engine)
print("Loaded from MySQL!")

Loaded from MySQL!


In [2]:
# Total order value = items price + freight, per order
order_value = (order_items.groupby("order_id")
               .agg(items_total=("price", "sum"),
                    freight_total=("freight_value", "sum"),
                    item_count=("order_item_id", "count"))
               .reset_index())
order_value["total_order_value"] = order_value["items_total"] + order_value["freight_total"]

orders = orders.merge(order_value, on="order_id", how="left")

# Delivery days and delay (only meaningful for delivered orders)
orders["delivery_days"] = (orders["order_delivered_customer_date"]
                           - orders["order_purchase_timestamp"]).dt.days
orders["delivery_delay_days"] = (orders["order_delivered_customer_date"]
                                 - orders["order_estimated_delivery_date"]).dt.days
orders["is_delayed"] = orders["delivery_delay_days"] > 0

orders[["order_id", "total_order_value", "delivery_days", "delivery_delay_days", "is_delayed"]].head()

,order_id,total_order_value,delivery_days,delivery_delay_days,is_delayed
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,7.0,-9.0,False
1,00018f77f2f0320c557190d7a144bdd3,259.83,16.0,-3.0,False
2,000229ec398224ef6ca0657da4fc703e,216.87,7.0,-14.0,False
3,00024acbcdf0a6daa1e931b038114c75,25.78,6.0,-6.0,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,25.0,-16.0,False


In [3]:
delivered = orders[orders["order_status"] == "delivered"]
print("Average delivery days:", round(delivered["delivery_days"].mean(), 1))
print("Delayed orders:", delivered["is_delayed"].sum())
print("Delayed %:", round(delivered["is_delayed"].mean() * 100, 1))

Average delivery days: 12.1
Delayed orders: 6534
Delayed %: 6.8


In [4]:
# Attach the real customer identity to each order
orders = orders.merge(
    customers[["customer_id", "customer_unique_id", "customer_state"]],
    on="customer_id", how="left"
)

# Only count orders that were not canceled/unavailable
valid_orders = orders[~orders["order_status"].isin(["canceled", "unavailable"])]

customer_features = (valid_orders.groupby("customer_unique_id")
    .agg(customer_order_count=("order_id", "nunique"),
         customer_total_spending=("total_order_value", "sum"))
    .reset_index())

customer_features["avg_order_value"] = (
    customer_features["customer_total_spending"] / customer_features["customer_order_count"]
)
customer_features["is_repeat_customer"] = customer_features["customer_order_count"] > 1

print(customer_features.shape)
print("Repeat customers:", customer_features["is_repeat_customer"].sum())
customer_features.head()

(94990, 5)
Repeat customers: 2888


,customer_unique_id,customer_order_count,customer_total_spending,avg_order_value,is_repeat_customer
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,False
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,False
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,False
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,False
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,False


In [5]:
seller_features = (order_items.groupby("seller_id")
    .agg(seller_revenue=("price", "sum"),
         seller_order_count=("order_id", "nunique"),
         seller_items_sold=("order_item_id", "count"))
    .reset_index())

seller_features.sort_values("seller_revenue", ascending=False).head()

,seller_id,seller_revenue,seller_order_count,seller_items_sold
857,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132,1156
1013,53243585a1d6dc2643021fd1853d8905,222776.05,358,410
881,4a3ca9315b744ce9f8e9374361493884,200472.92,1806,1987
3024,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585,586
1535,7c67e1448b00f6e969d365cea6b010ab,187923.89,982,1364
